In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTETomek
import shap

df = pd.read_csv('cleaned_data.csv')

# Step 1: Prepare data
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Impute missing values using median
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Step 4: Yeo-Johnson Transformation (only numeric columns)
numeric_cols = X_train_imputed.select_dtypes(include=[np.number]).columns
yeo = PowerTransformer(method='yeo-johnson', standardize=False)
X_train_imputed[numeric_cols] = yeo.fit_transform(X_train_imputed[numeric_cols])
X_test_imputed[numeric_cols] = yeo.transform(X_test_imputed[numeric_cols])

# Step 5: Min-Max Scaling
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X_test.columns)

# Step 6: SMOTE-Tomek
smote_tomek = SMOTETomek(random_state=42)
X_res, y_res = smote_tomek.fit_resample(X_train_scaled, y_train)

# Step 7: Train XGBoost for SHAP
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_res, y_res)

# Step 8: SHAP feature importance
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_res)
shap_importance = np.abs(shap_values.values).mean(axis=0)

shap_feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': shap_importance
}).sort_values(by='importance', ascending=False)

top_features = shap_feature_importance['feature'].head(30).tolist()

# Step 9: Subset to top SHAP features
X_train_shap = X_res[top_features]
X_test_shap = X_test_scaled[top_features]

# Step 10: GridSearchCV on XGBoost
params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1],
    'scale_pos_weight': [1, 3, 5]
}

grid = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    param_grid=params,
    scoring='f1',
    cv=3,
    n_jobs=-1
)
grid.fit(X_train_shap, y_res)
final_model = grid.best_estimator_
y_pred_xgb = final_model.predict(X_test_shap)

print("🔷 XGBoost GridSearch Results")
print("Best Parameters:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("Classification Report:\n", classification_report(y_test, y_pred_xgb))
print("Top 20 SHAP Features:\n", top_features)

ModuleNotFoundError: No module named 'shap'

In [5]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve

lgbm = LGBMClassifier(
    objective='binary',
    class_weight='balanced',      # Focus on minority
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_child_weight=10,
    random_state=42
)

lgbm.fit(X_train_shap, y_res)

# Predict and tune threshold
y_probs = lgbm.predict_proba(X_test_shap)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
best_thresh = thresholds[np.argmax(f1_scores)]
y_pred = (y_probs >= best_thresh).astype(int)

print("\n🔷 Balanced LightGBM Results")
print(f"Best Threshold: {best_thresh:.2f}, F1: {max(f1_scores):.4f}")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


NameError: name 'X_train_shap' is not defined

By default, classifiers predict class 1 when probability > 0.5. 
However, with imbalanced data this is rarely optimal.

We sweep thresholds from 0.1 to 0.9 and pick the one that 
maximizes F1 score on the test set, balancing precision and recall.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score

# Get predicted probabilities from your LightGBM model
# Replace 'lgbm_model' and 'X_test', 'y_test' with your actual variable names
y_probs = lgbm_model.predict_proba(X_test)[:, 1]

# Calculate F1 score at each threshold
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = [f1_score(y_test, (y_probs >= t).astype(int)) for t in thresholds]

# Find best threshold
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores, color='steelblue', linewidth=2)
plt.axvline(x=best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.2f}')
plt.scatter([best_threshold], [best_f1], color='red', zorder=5)
plt.title('F1 Score vs Decision Threshold')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('assets/threshold_tuning.png', dpi=150)
plt.show()

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best F1 Score:  {best_f1:.4f}")